In [ ]:
%cd ~/thesis/MSAE

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# general imports
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm
import seaborn as sns
import matplotlib.pyplot as plt
import os


In [ ]:
# imports from this repo
from my_config import CONFIGS
from path_hub import PathBuilder

In [ ]:
foundation_model = "vit"
inference_dataset_str = "cc3m"
split = "validation"
configs_dict = {key:value for key, value in CONFIGS.items() if inference_dataset_str in key and foundation_model in key}
configs_dict.keys() 

In [ ]:
def get_filename(inference_dataset_str, split):
    if inference_dataset_str == "cc3m":
        if split == "train":
            dataset_str = "cc3m_ViT-L~14_train_image_2820737_768"
        elif split == "validation":
            dataset_str = "cc3m_ViT-L~14_validation_image_13002_768"
    elif inference_dataset_str == "imagenet":
        if split == "train":
            dataset_str = "imagenet_ViT-L~14_train_image_1281167_768"
        elif split == "validation":
            dataset_str = "imagenet_ViT-L~14_validation_image_50000_768"
    else:
        raise ValueError(f"Unknown dataset string: {inference_dataset_str}"
        )
    if foundation_model == "dino":
        dataset_str = dataset_str.replace("ViT-L~14", "dinov2-base")
    filename = f"{dataset_str}_metrics.csv"
    return filename

In [ ]:


dataset_name = inference_dataset_str

filename = get_filename(dataset_name, split)
dfs = []
for config_name, config in configs_dict.items():
    path = os.path.join(PathBuilder(config=config).get_standard_sae_metrics_path(), filename)
    print(f"Loading metrics from {path}...")
    if not os.path.exists(path):
        print(f"File {path} does not exist. Skipping...")
        continue
    df_temp = pd.read_csv(path)
    df_temp['config'] = config_name  # Add config name as a column
    dfs.append(df_temp)

# Combine all dataframes
df = pd.concat(dfs, ignore_index=True)
df.head(10)

In [ ]:
# Create a clean display of metrics with mean ± std
def format_mean_std(mean, std):
    """Format mean and std into a readable string"""
    #return f"{mean:.4f}"
    return f"{mean:.4f} ± {std:.4f}"

In [ ]:

# Select metrics to display (only sparse, excluding metadata columns)
metric_columns = [col for col in df.columns if col not in
                ['timestamp', 'model_name', 'model_path', 'data_name',
                'data_path', 'num_samples', 'seed', 'config'] and "sparse" in col]

mean_cols = [col for col in metric_columns if col.endswith('_mean')]
metrics_to_display = [col.replace('_mean', '') for col in mean_cols]

# Display names for SAE types (maps config name substrings to human-readable names)
SAE_TYPE_DISPLAY_NAMES = {
    'topk_k': 'TopKSAE',
    'archmsae_uw': 'ArchMSAE',
    'msae_rw': 'ActMSAE',
    #"hsae_361_16_6144_cc3m": "HSAE",
    "361_16_v2": "HSAE-v2",
    "ewgsae_x8": "EWGSAE",
    'mpsae': 'MPSAE',
}

def get_display_name(config_name):
    for key, display_name in SAE_TYPE_DISPLAY_NAMES.items():
        if key in config_name:
            return display_name
    return config_name

# Group rows by SAE type (collapsing across seeds), compute mean and std across seeds
df['sae_type'] = df['config'].apply(get_display_name)
grouped = df.groupby('sae_type')[mean_cols]
cross_seed_means = grouped.mean()
cross_seed_stds = grouped.std(ddof=1)

display_data = {
    metric: {
        sae_type: format_mean_std(cross_seed_means.loc[sae_type, mean_col],
                                   cross_seed_stds.loc[sae_type, mean_col])
        for sae_type in cross_seed_means.index
    }
    for metric, mean_col in zip(metrics_to_display, mean_cols)
}

display_df = pd.DataFrame(display_data)
display_df.index.name = 'model'
display(display_df)

# export as csv
output_csv_path = f"reconstruction_metrics_{dataset_name}_{split}.csv"
display_df.to_csv(output_csv_path)


In [ ]:

# only include sparse columns
sparse_metric_columns = [col for col in display_df.columns if col.startswith('sparse_')]
display_df_sparse = display_df[sparse_metric_columns]

# remove "sparse_" prefix, split on underscores and capitalize words
display_df_sparse.columns = [col.replace('sparse_', '').replace('_', ' ').title() for col in display_df_sparse.columns]
display(display_df_sparse)

SELECTED_COLUMNS = ["Fvu", "Cosine Similarity", "L0"] #, "Cknna"]
display_df_sparse = display_df_sparse[SELECTED_COLUMNS]
display(display_df_sparse)

# turn into latex table
latex_table = display_df_sparse.to_latex()
print(latex_table)
